# Gemini scoring of listing descriptions

Each listing description is sent to Gemini with a fixed rubric, and the model returns five scores as JSON. The scores are saved to `data/gemini_encoded_descriptions.csv`, keyed by `zpid`, and `model/xgboost.ipynb` joins them back in.

Gemini sees **only the description text**. Web search grounding is off, so the model can't look up a listing or what it sold for. Set `GOOGLE_API_KEY` in `.env` before running.

In [ ]:
import pandas as pd

# Load the pipeline output; zpid is the key used to join these features back in model/xgboost.ipynb
df = pd.read_csv('../data/final_dataset.csv')

zpids = df['zpid'].tolist()
descriptions = df['basic_info_description'].fillna('').astype(str)

In [ ]:
import asyncio
import concurrent.futures
import os

import dotenv
import nest_asyncio
from google import genai
from google.genai import types
from tqdm import tqdm

dotenv.load_dotenv()
nest_asyncio.apply()

MODEL = "gemini-flash-lite-latest"
MAX_CONCURRENCY = 10

SYSTEM_PROMPT = "You are a Real Estate Quantitative Analyst. Your task is to analyze property descriptions and convert qualitative language into strict numerical values for a regression machine learning model.\n\nConstraints:\n1. Be Objective: Ignore marketing fluff (\"breathtaking,\" \"stunning\") unless it correlates with material value.\n2. Be Cynical: Assume \"cozy\" means small, and \"TLC\" means expensive repairs.\n3. Output Format: You must output valid JSON only. No markdown formatting, no conversational text.\n\nScoring Rubric (Strict Adherence Required):\n\n1. luxury_score (Integer 1-10)\nReflects the quality of finishes and materials.\n* 1-3 (Low): Laminate counters, linoleum/carpet, builder-grade cabinets, \"potential.\"\n* 4-6 (Mid): Standard stainless steel, some hardwood, \"well maintained,\" \"clean.\"\n* 7-8 (High): Granite/Quartz, hardwood throughout, crown molding, \"gourmet kitchen.\"\n* 9-10 (Premium): Sub-Zero/Viking appliances, imported stone, smart home, custom millwork, \"architectural masterpiece.\"\n\n2. renovation_index (Integer 0-3)\nReflects the recency and extent of updates.\n* 0 (Vintage/Gut): \"TLC,\" \"Handyman special,\" \"Needs work,\" \"Original condition.\"\n* 1 (Dated): Liveable but outdated style (e.g., 1990s oak cabinets, older tile).\n* 2 (Updated): \"New appliances,\" \"Updated bath,\" \"New flooring\" (Partial updates).\n* 3 (Turn-Key): \"Fully remodeled,\" \"Gut renovation,\" \"New Construction,\" \"Everything new.\"\n\n3. distress_probability (Float 0.0 - 1.0)\nThe likelihood the seller is desperate or the asset is distressed.\n* 0.0: Standard listing.\n* 0.3: \"Relocating,\" \"Must sell.\"\n* 0.7: \"As-is,\" \"Cash only,\" \"Investor special.\"\n* 1.0: \"Foreclosure,\" \"Bank owned,\" \"Short sale,\" \"Court ordered.\"\n\n4. marketing_hype_ratio (Float 0.0 - 1.0)\nThe ratio of subjective adjectives to objective facts.\n* 0.1: Dry, factual description (e.g., \"3 bed, 2 bath, new roof\").\n* 0.9: Pure fluff (e.g., \"Feel the magic of this breathtaking sanctuary...\").\n\n5. location_premium_inferred (Integer 0-2)\nDoes the text mention specific high-value location keywords?\n* 0: No specific location highlights.\n* 1: \"Corner lot,\" \"Cul-de-sac,\" \"Quiet street.\"\n* 2: \"Waterfront,\" \"Ocean view,\" \"Golf course,\" \"City skyline view.\""

# No tools: the score must come from the description alone, not from a web search for the listing
CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM_PROMPT,
    temperature=0,
    response_mime_type="application/json",
    thinking_config=types.ThinkingConfig(thinking_budget=0),
)


def score_description(client, description):
    """Score one description; returns the raw JSON text, or an error object as JSON text."""
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=f"Encode this description: {description}",
            config=CONFIG,
        )
        return response.text
    except Exception as e:
        print(f"[API ERROR] {e}")
        return '{"error": "request failed"}'


async def score_all(descriptions):
    client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    # The SDK call is blocking, so run it in a thread pool sized to the concurrency limit
    loop = asyncio.get_running_loop()
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=MAX_CONCURRENCY)
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)
    results = [None] * len(descriptions)
    pbar = tqdm(total=len(descriptions), unit="desc")

    async def score(i, desc):
        # Nothing to score: leave the row empty instead of letting the model invent values
        if desc.strip():
            async with semaphore:
                results[i] = await loop.run_in_executor(executor, score_description, client, desc)
        pbar.update(1)

    await asyncio.gather(*(score(i, d) for i, d in enumerate(descriptions)))
    pbar.close()
    executor.shutdown(wait=True)
    return results


all_results = asyncio.run(score_all(list(descriptions)))

In [ ]:
import json
import pandas as pd

FEATURES = ['luxury_score', 'renovation_index', 'distress_probability',
            'marketing_hype_ratio', 'location_premium_inferred']

def parse_result(r):
    """Parse one model response into a dict of features; anything unparseable becomes empty."""
    if r is None:
        return {}
    if isinstance(r, dict):
        return r
    text = r.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as e:
        print(f"Could not parse row: {r[:200]}\nError: {e}")
        return {}
    return parsed if isinstance(parsed, dict) and 'error' not in parsed else {}

gemini_df = pd.DataFrame([parse_result(r) for r in all_results]).reindex(columns=FEATURES)
gemini_df.insert(0, 'zpid', zpids)

gemini_df.to_csv('../data/gemini_encoded_descriptions.csv', index=False)
print(f"Saved {len(gemini_df)} rows ({gemini_df[FEATURES[0]].isna().sum()} empty) to ../data/gemini_encoded_descriptions.csv")